In [1]:
import sys
sys.executable

'c:\\Users\\mauro\\Downloads\\proyecto_ml_data-main\\.venv\\Scripts\\python.exe'

In [2]:
import pandas as pd
import numpy as np
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

pandas: 3.0.1
numpy: 2.4.2


Rutas y librerías

In [3]:
import os
import pandas as pd
import numpy as np

# Rutas (notebook está en /notebooks)
RAW_DIR = "../data/raw/store_sales"
INTERIM_DIR = "../data/interim/store_sales"
PROCESSED_DIR = "../data/processed/store_sales"

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(INTERIM_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

RAW_TRAIN = os.path.join(RAW_DIR, "train.parquet")
RAW_TEST  = os.path.join(RAW_DIR, "test.parquet")

print("RAW_TRAIN:", RAW_TRAIN, "exists:", os.path.exists(RAW_TRAIN))
print("RAW_TEST :", RAW_TEST,  "exists:", os.path.exists(RAW_TEST))

RAW_TRAIN: ../data/raw/store_sales\train.parquet exists: True
RAW_TEST : ../data/raw/store_sales\test.parquet exists: True


Cargar Parquet

In [4]:
train = pd.read_parquet(RAW_TRAIN)
test  = pd.read_parquet(RAW_TEST)

print("train shape:", train.shape)
print("test  shape:", test.shape)
train.head()

train shape: (3000888, 6)
test  shape: (28512, 6)


,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


Revisión rápida

In [5]:
train.info()
train.isna().sum().head(20)

<class 'pandas.DataFrame'>
RangeIndex: 3000888 entries, 0 to 3000887
Data columns (total 6 columns):
 #   Column       Dtype  
---  ------       -----  
 0   id           int64  
 1   date         str    
 2   store_nbr    int64  
 3   family       str    
 4   sales        float64
 5   onpromotion  int64  
dtypes: float64(1), int64(3), str(2)
memory usage: 196.8 MB


id             0
date           0
store_nbr      0
family         0
sales          0
onpromotion    0
dtype: int64

TRANSFORM (limpieza + tipos + features)

In [6]:
df = train.copy()

# Tipos
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Asegurar numéricos
df["store_nbr"] = pd.to_numeric(df["store_nbr"], errors="coerce").astype("Int64")
df["sales"] = pd.to_numeric(df["sales"], errors="coerce")
df["onpromotion"] = pd.to_numeric(df["onpromotion"], errors="coerce").fillna(0).astype(int)

# Limpiar nulos críticos
df = df.dropna(subset=["date", "store_nbr", "sales"])

# Features de calendario
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["weekday"] = df["date"].dt.weekday  # 0=Lunes ... 6=Domingo

# Chequeo rápido
print("Transform OK - shape:", df.shape)
df.head()

Transform OK - shape: (3000888, 10)


,id,date,store_nbr,family,sales,onpromotion,year,month,day,weekday
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,2013,1,1,1
1,1,2013-01-01,1,BABY CARE,0.0,0,2013,1,1,1
2,2,2013-01-01,1,BEAUTY,0.0,0,2013,1,1,1
3,3,2013-01-01,1,BEVERAGES,0.0,0,2013,1,1,1
4,4,2013-01-01,1,BOOKS,0.0,0,2013,1,1,1


LOAD (dataset final para forecasting)

Para forecasting, lo normal es trabajar con una serie temporal agregada.
Creamos un dataset “canónico” por fecha + tienda + familia:

In [7]:
# Agregación para forecasting
agg = (
    df.groupby(["date", "store_nbr", "family"], as_index=False)
      .agg(
          sales=("sales", "sum"),
          onpromotion=("onpromotion", "sum")
      )
)

# Re-crear features de calendario ya sobre la tabla agregada
agg["year"] = agg["date"].dt.year
agg["month"] = agg["date"].dt.month
agg["weekday"] = agg["date"].dt.weekday

print("Processed shape:", agg.shape)
agg.head()

Processed shape: (3000888, 8)


,date,store_nbr,family,sales,onpromotion,year,month,weekday
0,2013-01-01,1,AUTOMOTIVE,0.0,0,2013,1,1
1,2013-01-01,1,BABY CARE,0.0,0,2013,1,1
2,2013-01-01,1,BEAUTY,0.0,0,2013,1,1
3,2013-01-01,1,BEVERAGES,0.0,0,2013,1,1
4,2013-01-01,1,BOOKS,0.0,0,2013,1,1


Guardar PROCESSED (fin de Load)

In [8]:
PROCESSED_PATH = os.path.join(PROCESSED_DIR, "sales_processed.parquet")
agg.to_parquet(PROCESSED_PATH, index=False)

print("✅ Guardado:", PROCESSED_PATH)

✅ Guardado: ../data/processed/store_sales\sales_processed.parquet


In [9]:
print(train.shape, test.shape)
print(df.shape)
print(agg.shape)

(3000888, 6) (28512, 6)
(3000888, 10)
(3000888, 8)
